# Stage 1: Generate Descriptions

This notebook generates Stage 1 training data using GPT-4.1.

For each TEM subfigure, GPT receives the image and its parent figure caption, and generates two types of descriptions:

- **VisionGround** — describes only visually observable features using generic visual language, without domain-specific terminology
- **DomainContext** — grounds domain-specific terms from the caption in visible image features, without inferring invisible chemistry or mechanisms

# Import packages

In [1]:
import os
import gc
import pandas as pd
import base64
import csv
from pathlib import Path
from openai import OpenAI

## Settings

Adjust the paths below to match your local setup:

- `FIGURE_DIR`: directory containing TEM subfigures (output of `subfigure_extraction.ipynb`)
- `INPUT_CSV_PATH`: CSV containing subfigure metadata with `CROP_IMAGE` and `CAPTION` columns
- `OUTPUT_CSV_PATH`: output CSV where generated descriptions will be saved

The OpenAI API key is read from the `OPENAI_API_KEY` environment variable.

In [ ]:
FIGURE_DIR      = "/path/to/your/TEM_figures"
INPUT_CSV_PATH  = "/path/to/your/total_dataset.csv"
OUTPUT_CSV_PATH = "/path/to/your/stage1_descriptions.csv"

API_Key = os.getenv("OPENAI_API_KEY")
client  = OpenAI(api_key=API_Key)

## Define Prompts

In [3]:
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

In [ ]:
def system_prompt():
    return """
You are a scientific vision-language data generator for multimodal model training.

Your task is to describe TEM image crops under strict grounding constraints.

GLOBAL RULES:
1. Visual evidence is the primary source of truth.
2. Captions provide domain context but are NOT direct visual evidence.
3. Do NOT fabricate materials, mechanisms, synthesis details, or numeric values not visually supported.
4. Domain-specific terminology is allowed ONLY in DomainContext mode.
5. VisionGround descriptions must use generic visual language only.
6. Do not mention the caption explicitly in outputs.
7. Output must strictly follow the JSON schema.
"""

In [ ]:
def build_user_prompt(caption):
    return f"""
You will receive:
1) A TEM image cropped from the parent figure
2) A scientific caption describing the parent figure

Generate two types of descriptions of the TEM crop.

----------------------
MODE A — VisionGround
----------------------
Describe ONLY observable visual features using generic visual language.
Do NOT use scientific terms, polymer names, or mechanisms.

----------------------
MODE B — DomainContext
----------------------
This description must STILL be grounded in visible image features.

You may use domain-specific morphology terms appearing in the caption,
but ONLY to label structures that are visually observable in the TEM image.

Do NOT infer synthesis methods, mechanisms, or invisible chemistry.

----------------------
### STEP-BY-STEP REASONING PROTOCOL (Internal Thought Process):
Before generating the output, you must perform a visual audit in the "reasoning_process" field:

1. **Visual Scan**: List raw visual features (contrast, shape, density, distribution).
2. **Scale Audit**: Is a scale bar visible? If YES, use it for size estimation. If NO, don't mention about absolute dimensions.
3. **Caption Cross-Check**: Map words from the caption to specific visual landmarks. Identify "Caption claims" vs. "Visual evidence."
4. **Absence Identification**: List items mentioned in the caption that are MISSING or INVISIBLE in this crop. Do not hallucinate their presence.

----------------------
Return JSON with this schema:
{{
  "reasoning_process": "Step-by-step audit of visual features vs caption claims.",
  "visionground_brief": "...",
  "visionground_detailed": "...",
  "domaincontext_brief": "...",
  "domaincontext_detailed": "...",
}}

Caption:
\"\"\"
{caption}
\"\"\"
"""

## Check Progress

Loads already-processed images from `OUTPUT_CSV_PATH` so the notebook can be safely resumed after interruption.

In [ ]:
done_images = set()
if os.path.exists(OUTPUT_CSV_PATH):
    with open(OUTPUT_CSV_PATH, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        for row in reader:
            if row:
                done_images.add(row[0])
print(f"{len(done_images)} images done")

## Generate Descriptions

For each subfigure in `INPUT_CSV_PATH`:
1. Skips images already processed
2. Encodes the image in base64 and sends it to GPT-4.1 with the parent caption
3. Saves the response incrementally to `OUTPUT_CSV_PATH`

In [ ]:
df = pd.read_csv(INPUT_CSV_PATH, dtype=str, encoding="utf-8")

file_exists = os.path.exists(OUTPUT_CSV_PATH)

with open(OUTPUT_CSV_PATH, "a", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    if not file_exists:
        writer.writerow(["CROP_IMAGE", "RESPONSE"])
    
    for idx, row in df.iterrows():
        
        image_name = row['CROP_IMAGE']

        if image_name in done_images:
            continue
    
        print(f"Dealing with {idx}th image: {row['CROP_IMAGE']} ")
        
        image_path = FIGURE_DIR / image_name
        caption = row['CAPTION']
        base64_image = encode_image(image_path)

        try:
            response = client.responses.create(
                model="gpt-4.1",
                temperature=0.25,
                input=[
                    {
                        "role": "system",
                        "content": [{"type": "input_text", "text": system_prompt()}],
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "input_text", "text": build_user_prompt(caption)},
                            {
                                "type": "input_image",
                                "image_url": f"data:image/png;base64,{base64_image}",
                            },
                        ],
                    },
                ],
            )

            output = response.output_text
            writer.writerow([row['CROP_IMAGE'],output])
            f.flush()
            os.fsync(f.fileno())

            done_images.add(image_name)
            print(f"Save: {image_name}")
        except Exception as e:
            print(f"Error on {image_name}: {e}")
            continue
            
        del base64_image
        del response
        gc.collect()